In [1]:
# Kiểm tra tính nhất quán của dữ liệu ngày mà 
# lịch sử đều đồng bộ với node loc

# Thư viện

In [2]:
import osmnx as ox

import pandas as pd
import numpy as np

from pathlib import Path

import requests
import json

from rapidfuzz import fuzz
from unidecode import unidecode

# Load các file csv

In [3]:
PATH = Path("../data/raw")

files = {
    "nodes": PATH / "nodes.csv",
    "streets": PATH / "streets.csv",
    "segments": PATH / "segments.csv",
    "status": PATH / "segment_status.csv",
    "train": PATH / "train.csv"
}

In [4]:
nodes_df = pd.read_csv(files["nodes"])
streets_df = pd.read_csv(files["streets"])
segments_df = pd.read_csv(files["segments"])
status_df = pd.read_csv(files["status"])
train_df = pd.read_csv(files["train"])

In [5]:
train_nodes = list(set(train_df["s_node_id"]) | set(train_df["e_node_id"]))

In [6]:
train_streets = set(train_df["street_id"].to_list())

# Load graph

In [7]:
with open("../data/raw/osm_train_2019_01_03.json", "r", encoding="utf-8") as f:
    osm_data = json.load(f)

In [8]:
osm_data.keys()

dict_keys(['version', 'generator', 'osm3s', 'elements'])

In [9]:
osm_data["version"]

0.6

## OSM Element

In [10]:
osm_elements = pd.DataFrame(osm_data["elements"])

In [11]:
osm_elements.head()

,type,id,lat,lon,tags,nodes,members
0,node,366367322,10.799155,106.657136,NaN,NaN,NaN
1,node,366367392,10.775732,106.614032,NaN,NaN,NaN
2,node,366367450,10.753841,106.645673,NaN,NaN,NaN
3,node,366367451,10.793792,106.695366,NaN,NaN,NaN
4,node,366367839,10.807689,106.664528,NaN,NaN,NaN


In [12]:
osm_elements["type"].unique()

array(['node', 'way', 'relation'], dtype=object)

## OSM Node

In [13]:
osm_nodes_df = osm_elements[osm_elements["type"] == "node"]

In [14]:
osm_nodes_df.head()

,type,id,lat,lon,tags,nodes,members
0,node,366367322,10.799155,106.657136,NaN,NaN,NaN
1,node,366367392,10.775732,106.614032,NaN,NaN,NaN
2,node,366367450,10.753841,106.645673,NaN,NaN,NaN
3,node,366367451,10.793792,106.695366,NaN,NaN,NaN
4,node,366367839,10.807689,106.664528,NaN,NaN,NaN


## OSM Way

In [15]:
osm_way_df = osm_elements[osm_elements["type"] == "way"]

In [16]:
print(osm_way_df.shape)
osm_way_df.head()

(7606, 7)


,type,id,lat,lon,tags,nodes,members
3061,way,32575768,NaN,NaN,"{'name': 'Đường số 27', 'highway': 'residential'}","[366369613, 5795144851, 366418963, 366373068, ...",NaN
3062,way,32576350,NaN,NaN,"{'name': 'Đường số 18', 'highway': 'residential'}","[366372346, 5755079612, 3040292134, 5755079614...",NaN
3063,way,32576691,NaN,NaN,"{'addr:city': 'Ho Chi Minh', 'addr:district': ...","[366452320, 3351962143, 3351962141, 366375776,...",NaN
3064,way,32576911,NaN,NaN,"{'highway': 'residential', 'name': 'Tân Thành'}","[5778381635, 5552002921, 4878713065, 580288880...",NaN
3065,way,32577060,NaN,NaN,{'highway': 'residential'},"[366371080, 366389438, 5735589498, 5735589492,...",NaN


In [17]:
data = []

for _, row in osm_way_df.iterrows():
    way_id = row['id']
    nodes = row['nodes']
    if isinstance(nodes, list) and len(nodes) >= 2:
        for i in range(len(nodes)-1):
            data.append({
                'way_id': way_id,
                's_node_id': nodes[i],      # from
                'e_node_id': nodes[i+1]     # to
            })

osm_edges_df = pd.DataFrame(data)
print(osm_edges_df.head())

     way_id   s_node_id   e_node_id
0  32575768   366369613  5795144851
1  32575768  5795144851   366418963
2  32575768   366418963   366373068
3  32575768   366373068  5753228946
4  32575768  5753228946  5795144815


In [18]:
osm_reverse_edges_df = osm_edges_df.rename(columns={
    "s_node_id":"e_node_id", 
    "e_node_id":"s_node_id"
})
osm_undirected_edges_df = pd.concat([osm_edges_df, osm_reverse_edges_df])
print(osm_undirected_edges_df.shape)
osm_undirected_edges_df.head()

(112706, 3)


,way_id,s_node_id,e_node_id
0,32575768,366369613,5795144851
1,32575768,5795144851,366418963
2,32575768,366418963,366373068
3,32575768,366373068,5753228946
4,32575768,5753228946,5795144815


In [19]:
osm_way_tags_df = pd.json_normalize(
    osm_way_df["tags"]
        .where(
            osm_way_df["tags"]
                .notna(), 
            other=[{}]
        )
)

osm_way_tags_df.insert(0, "way_id", osm_way_df["id"].to_numpy())
osm_way_tags_df.head()

,way_id,name,highway,addr:city,addr:district,name:en,ref,addr:subdistrict,service,oneway,...,name:ja,name:th,fixme,traffic_signals,lay,information,covered,motorcar:forward,crossing,footway
0,32575768,Đường số 27,residential,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,32576350,Đường số 18,residential,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,32576691,Phan Văn Hớn,secondary,Ho Chi Minh,Hoc Mon,Phan Van Hon,14,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,32576911,Tân Thành,residential,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,32577060,NaN,residential,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Kiểm tra tính nhất quán

## Node

### Node id

In [20]:
osm_nodes = set(osm_nodes_df["id"])
train_nodes = set(train_df["s_node_id"]) | set(train_df["e_node_id"])

In [21]:
inter_train_osm_nodes = osm_nodes & train_nodes

In [22]:
print(len(osm_nodes))
print(len(train_nodes))
print(len(inter_train_osm_nodes))

53115
11314
11314


### Node location

In [23]:
osm_node_locs = set(osm_nodes_df[["id", "lon", "lat"]].apply(tuple, axis=1))
train_node_locs = set(train_df[["s_node_id", "long_snode", "lat_snode"]].apply(tuple, axis=1)) | \
                    set(train_df[["e_node_id", "long_enode", "lat_enode"]].apply(tuple, axis=1))

In [24]:
inter_osm_train_node_locs = osm_node_locs & train_node_locs
print(len(osm_node_locs))
print(len(train_node_locs))
print(len(inter_osm_train_node_locs))

53115
11314
11314


## Way

### Way id

In [25]:
train_segment_ids = set(train_df["street_id"])
osm_ways_ids = set(osm_way_df["id"])
inter_osm_train_edge = osm_ways_ids & train_segment_ids

print(len(train_segment_ids))
print(len(osm_ways_ids))
print(len(inter_osm_train_edge))

1967
7606
1967


### Way nodes

In [26]:
train_segment_nodes = set(train_df[["street_id", "s_node_id", "e_node_id"]].apply(tuple, axis=1))
osm_way_nodes = set(osm_undirected_edges_df.apply(tuple, axis=1))
inter_osm_train_edges = osm_way_nodes & train_segment_nodes

print(len(train_segment_nodes))
print(len(osm_way_nodes))
print(len(inter_osm_train_edges))

10027
112706
10027


# Lưu relation

In [29]:
osm_relation_df = osm_elements[osm_elements["type"] == "relation"]
osm_relation_df = osm_relation_df.dropna(how="all", axis=1)
print(osm_relation_df.shape)
print(osm_relation_df.columns)
osm_relation_df.head()

(63, 4)
Index(['type', 'id', 'tags', 'members'], dtype='object')


,type,id,tags,members
5631,relation,2922023,"{'hour_off': '18:00', 'hour_on': '17:00', 'typ...","[{'type': 'node', 'ref': 366443898, 'role': 'v..."
5632,relation,3423384,"{'except': 'motorcycle', 'restriction': 'no_le...","[{'type': 'node', 'ref': 2079964894, 'role': '..."
7974,relation,2907508,"{'type': 'restriction', 'restriction': 'no_lef...","[{'type': 'node', 'ref': 366476538, 'role': 'v..."
7975,relation,2914954,"{'restriction': 'no_u_turn', 'type': 'restrict...","[{'type': 'node', 'ref': 366462455, 'role': 'v..."
7976,relation,2970026,"{'except': 'motorcycle', 'type': 'restriction'...","[{'type': 'node', 'ref': 2079964991, 'role': '..."


In [ ]:
relation_members_df = 